# Fine-tune `yolo11s-seg.pt` on VehiDE

## 1. Setup

In [1]:
!pip install -q ultralytics psutil

import json, shutil, time, os, subprocess
from pathlib import Path

import torch

assert torch.cuda.is_available(), "Enable GPU in Settings → Accelerator before running."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

device_cap = torch.cuda.get_device_capability(0)
sm_str = f"sm_{device_cap[0]}{device_cap[1]}"
supported = torch.cuda.get_arch_list()
print(f"GPU compute capability: {device_cap} ({sm_str})")
print(f"This torch build's compiled kernel architectures: {supported}")

if sm_str not in supported:
    raise RuntimeError(
        f"This GPU's architecture ({sm_str}) has no compiled kernels in the "
        f"installed PyTorch build (supports: {supported}). Training will fail "
        f"immediately at model.to(device). Fix: change Settings -> Accelerator "
        f"to T4 and re-run from this cell."
    )
print("GPU/PyTorch build compatible.")

import psutil
ram_gb = psutil.virtual_memory().total / 1e9
ram_available_gb = psutil.virtual_memory().available / 1e9
print(f"System RAM: {ram_gb:.1f} GB total, {ram_available_gb:.1f} GB available")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.5 MB/s eta 0:00:00
GPU: Tesla T4
VRAM: 15.6 GB
GPU compute capability: (7, 5) (sm_75)
This torch build's compiled kernel architectures: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
GPU/PyTorch build compatible.
System RAM: 33.7 GB total, 31.8 GB available


## 2. Mount the converted segmentation dataset

In [2]:
!ls /kaggle/input/

SEG_ROOT = Path("/kaggle/input/datasets/m4rcuseryx/vehide-segmentation-dataset")
SEG_IMAGES_LABELS_ROOT = SEG_ROOT / "vehide_seg"
SEG_DATA_YAML_SOURCE = SEG_ROOT / "damage-seg.yaml"

assert SEG_ROOT.exists(), "Segmentation dataset not found at this path -- check /kaggle/input/ above and update SEG_ROOT."
assert SEG_IMAGES_LABELS_ROOT.exists(), f"Expected images/labels under {SEG_IMAGES_LABELS_ROOT}, not found."
assert SEG_DATA_YAML_SOURCE.exists(), f"Expected damage-seg.yaml at {SEG_DATA_YAML_SOURCE}, not found."

print(open(SEG_DATA_YAML_SOURCE).read())

CLASS_NAMES = ["dent", "scratch", "crack", "broken_lamp", "shattered_glass", "flat_tyre"]

train_n = len(list((SEG_IMAGES_LABELS_ROOT / "images" / "train").glob("*.jpg")))
val_n = len(list((SEG_IMAGES_LABELS_ROOT / "images" / "val").glob("*.jpg")))
print(f"Train images: {train_n}, Val images: {val_n}")

datasets
names:
- dent
- scratch
- crack
- broken_lamp
- shattered_glass
- flat_tyre
nc: 6
path: /kaggle/working/vehide_seg
test: images/test
train: images/train
val: images/val

Train images: 9545, Val images: 2047


In [3]:
import yaml

with open(SEG_DATA_YAML_SOURCE) as f:
    cfg = yaml.safe_load(f)
cfg["path"] = str(SEG_IMAGES_LABELS_ROOT)

SEG_DATA_YAML = "/kaggle/working/damage-seg.yaml"
with open(SEG_DATA_YAML, "w") as f:
    yaml.safe_dump(cfg, f)
print(open(SEG_DATA_YAML).read())

names:
- dent
- scratch
- crack
- broken_lamp
- shattered_glass
- flat_tyre
nc: 6
path: /kaggle/input/datasets/m4rcuseryx/vehide-segmentation-dataset/vehide_seg
test: images/test
train: images/train
val: images/val



## 3. Load the checkpoint - Model Loading

In [4]:
from ultralytics import YOLO

probe_model = YOLO("yolo11s-seg.pt")   # auto-downloads from Ultralytics' release on first use
print("Checkpoint's own class names (COCO, 80 classes -- not directly relevant, ")
print("since all heads are reinitialised for our 6-class task):")
print(probe_model.names)
print("\nTask:", probe_model.task)

n_params = sum(p.numel() for p in probe_model.model.parameters())
print(f"\nParameters: {n_params/1e6:.1f}M (compare: YOLOv8s-seg ~11.8M, YOLO11x-seg ~62.1M)")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Checkpoint's own class names (COCO, 80 classes -- not directly relevant, 
since all heads are reinitialised for our 6-class task):
{0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard',

## 4. Configuration

In [5]:
est_cache_ram_gb = train_n * (1280 * 1280 * 3) / 1e9
print(f"Estimated RAM cost of cache='ram' for the training split: ~{est_cache_ram_gb:.1f} GB")
print(f"Available RAM: {ram_available_gb:.1f} GB")

if est_cache_ram_gb < ram_available_gb * 0.6:
    CACHE_MODE = True
    print("-> Using cache=True (RAM). Comfortable headroom available.")
else:
    CACHE_MODE = "disk"
    print("-> RAM caching looks risky at this dataset size vs. available RAM. "
          "Using cache='disk' instead.")

Estimated RAM cost of cache='ram' for the training split: ~46.9 GB
Available RAM: 31.8 GB
-> RAM caching looks risky at this dataset size vs. available RAM. Using cache='disk' instead.


In [10]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")

auth_check = subprocess.run(
    ["kaggle", "datasets", "list", "-s", "zzz_auth_check_zzz", "-p", "1"],
    capture_output=True, text=True,
)
if auth_check.returncode != 0:
    print("STDOUT:", auth_check.stdout)
    print("STDERR:", auth_check.stderr)
    raise RuntimeError(
        "Kaggle API authentication failed. Checkpoint backups will NOT work "
        "until this is fixed. Check:\n"
        "  1. Add-ons -> Secrets -> confirm KAGGLE_USERNAME and KAGGLE_KEY are "
        "both added AND toggled ON for this notebook.\n"
        "  2. The values match your actual kaggle.json exactly.\n"
        "  3. Consider upgrading the kaggle package: !pip install -q -U kaggle"
    )
print("Kaggle API authentication OK.")

RUN_NAME = "yolo11s_seg_finetune"

SESSION_EPOCH_BUDGET = 999
BACKUP_EVERY_N_EPOCHS = 5
BACKUP_SLUG = "yolo11s-seg-finetune-checkpoint-backup" 
BACKUP_DATASET_ID = f"{os.environ['KAGGLE_USERNAME']}/{BACKUP_SLUG}"

BACKUP_STAGE = Path("/kaggle/working/checkpoint_backup_yolo11s")
BACKUP_STAGE.mkdir(parents=True, exist_ok=True)

TRAIN_ARGS = dict(
    data=SEG_DATA_YAML,
    epochs=20,
    imgsz=1280,
    batch=4,
    optimizer="AdamW",
    lr0=0.0005,
    lrf=0.01,
    cos_lr=True,
    weight_decay=0.0005,
    warmup_epochs=10,
    cls=0.3,
    dfl=1.7,
    dropout=0.1,
    multi_scale=False,
    patience=20,
    save_period=10,
    amp=True,
    cache=CACHE_MODE,
    workers=4,
    close_mosaic=10,
    seed=42,
    deterministic=True,
    plots=True,
    project="/kaggle/working/runs/yolo11s_seg",
)
print(json.dumps(TRAIN_ARGS, indent=2))

Kaggle API authentication OK.
{
  "data": "/kaggle/working/damage-seg.yaml",
  "epochs": 20,
  "imgsz": 1280,
  "batch": 4,
  "optimizer": "AdamW",
  "lr0": 0.0005,
  "lrf": 0.01,
  "cos_lr": true,
  "weight_decay": 0.0005,
  "warmup_epochs": 10,
  "cls": 0.3,
  "dfl": 1.7,
  "dropout": 0.1,
  "multi_scale": false,
  "patience": 20,
  "save_period": 10,
  "amp": true,
  "cache": "disk",
  "workers": 4,
  "close_mosaic": 10,
  "seed": 42,
  "deterministic": true,
  "plots": true,
  "project": "/kaggle/working/runs/yolo11s_seg"
}


## 5. Probe 

To determine approximate resources required - time and compute

In [8]:
import random

PROBE_DIR = Path("/kaggle/working/probe_subsample")
probe_img_dir = PROBE_DIR / "images" / "train"
probe_lbl_dir = PROBE_DIR / "labels" / "train"
probe_img_dir.mkdir(parents=True, exist_ok=True)
probe_lbl_dir.mkdir(parents=True, exist_ok=True)

all_train_imgs = sorted((SEG_IMAGES_LABELS_ROOT / "images" / "train").glob("*.jpg"))
sample = random.Random(42).sample(all_train_imgs, min(300, len(all_train_imgs)))
for img_path in sample:
    (probe_img_dir / img_path.name).symlink_to(img_path)
    lbl_path = SEG_IMAGES_LABELS_ROOT / "labels" / "train" / f"{img_path.stem}.txt"
    if lbl_path.exists():
        (probe_lbl_dir / lbl_path.name).symlink_to(lbl_path)

probe_cfg = dict(cfg)
probe_cfg["path"] = str(PROBE_DIR)
probe_cfg["train"] = "images/train"
probe_cfg["val"] = str(SEG_IMAGES_LABELS_ROOT / "images" / "val")

PROBE_YAML = "/kaggle/working/damage-seg-probe.yaml"
with open(PROBE_YAML, "w") as f:
    yaml.safe_dump(probe_cfg, f)
print(f"Probe subsample: {len(sample)} images")

Probe subsample: 300 images


In [9]:
probe_args = dict(TRAIN_ARGS)
probe_args["data"] = PROBE_YAML
probe_args["epochs"] = 1
probe_args["project"] = "/kaggle/working/runs/probe_yolo11s"
probe_args["name"] = "timing_probe"
probe_args["cache"] = False  

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
probe_run_model = YOLO("yolo11s-seg.pt")
probe_run_model.train(**probe_args)
probe_wall = time.time() - t0
peak_vram_gb = torch.cuda.max_memory_allocated() / 1e9
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"\nMeasured: {probe_wall/60:.1f} min for 1 epoch on {len(sample)} images.")
print(f"Peak VRAM used: {peak_vram_gb:.2f} GB / {total_vram_gb:.1f} GB available "
      f"({100*peak_vram_gb/total_vram_gb:.0f}%)")

if peak_vram_gb > total_vram_gb * 0.85:
    print("\nWARNING: peak VRAM usage is close to the card's limit even on this "
          "small probe. Lower TRAIN_ARGS['imgsz'] or ['batch'] in Section 4 and "
          "re-run this probe before continuing to Section 6.")
else:
    print("\nVRAM headroom looks OK for this config.")

full_epoch_est_min = (probe_wall / 60) * (train_n / len(sample))
print(f"\nEstimated time per epoch on the full {train_n}-image training set: "
      f"~{full_epoch_est_min:.1f} min")
print(f"Estimated total time for TRAIN_ARGS['epochs']={TRAIN_ARGS['epochs']}: "
      f"~{full_epoch_est_min * TRAIN_ARGS['epochs'] / 60:.1f} hours")
print("\nIf that total is impractical against your remaining Kaggle GPU quota, "
      "lower TRAIN_ARGS['epochs'] in Section 4 now and re-run that cell before "
      "continuing to Section 6.")

Ultralytics 8.4.110 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.3, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/damage-seg-probe.yaml, degrees=0.0, deterministic=True, device=, dfl=1.7, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=timing_probe, nbs=

## 6. Fine-tune, with multi-session checkpoint relay

In [11]:
meta_check = subprocess.run(
    ["kaggle", "datasets", "metadata", BACKUP_DATASET_ID, "-p", "/tmp/meta_check_yolo11s"],
    capture_output=True, text=True,
)
check = meta_check.returncode
resume_from = None

if check == 0:
    print(f"Found existing backup: {BACKUP_DATASET_ID} -- downloading...")
    os.makedirs("/kaggle/working/recovered_yolo11s", exist_ok=True)
    dl = subprocess.run(
        ["kaggle", "datasets", "download", BACKUP_DATASET_ID,
         "-p", "/kaggle/working/recovered_yolo11s", "--unzip", "-q"],
        capture_output=True, text=True,
    )
    if dl.returncode != 0:
        print(dl.stdout, dl.stderr)
        raise RuntimeError("Backup dataset metadata was found, but downloading it failed.")
    recovered_pt = list(Path("/kaggle/working/recovered_yolo11s").rglob("last.pt"))
    if recovered_pt:
        resume_from = recovered_pt[0]
        epoch_marker = Path("/kaggle/working/recovered_yolo11s/epoch.txt")
        print(f"Will resume from epoch {epoch_marker.read_text().strip() if epoch_marker.exists() else '?'}")
else:
    print(f"No existing backup dataset ({BACKUP_DATASET_ID}) -- session 1, starting fresh from yolo11s-seg.pt")
    print(f"(kaggle CLI returned: {meta_check.stderr.strip()[:200]})")

No existing backup dataset (m4rcuseryx/yolo11s-seg-finetune-checkpoint-backup) -- session 1, starting fresh from yolo11s-seg.pt
(kaggle CLI returned: )


In [ ]:
_session_start_epoch = {"value": None}
_dataset_exists = {"value": check == 0}
_backup_failures = []


def _push_backup(trainer, completed_epoch):
    last_pt = trainer.save_dir / "weights" / "last.pt"
    if not last_pt.exists():
        return
    shutil.copy(last_pt, BACKUP_STAGE / "last.pt")
    for extra in ("results.csv", "args.yaml"):
        p = trainer.save_dir / extra
        if p.exists():
            shutil.copy(p, BACKUP_STAGE / extra)
    (BACKUP_STAGE / "epoch.txt").write_text(str(completed_epoch))
    (BACKUP_STAGE / "dataset-metadata.json").write_text(json.dumps({
        "title": "YOLO11s-seg fine-tune checkpoint backup",
        "id": BACKUP_DATASET_ID,
        "licenses": [{"name": "CC0-1.0"}],
    }))

    if not _dataset_exists["value"]:
        result = subprocess.run(
            ["kaggle", "datasets", "create", "-p", str(BACKUP_STAGE), "--dir-mode", "zip", "-q"],
            capture_output=True, text=True,
        )
        if result.returncode == 0:
            _dataset_exists["value"] = True
    else:
        result = subprocess.run(
            ["kaggle", "datasets", "version", "-p", str(BACKUP_STAGE),
             "-m", f"epoch {completed_epoch}", "--dir-mode", "zip", "-q"],
            capture_output=True, text=True,
        )

    if result.returncode == 0:
        print(f"Backed up checkpoint at epoch {completed_epoch} -> {BACKUP_DATASET_ID}")
    else:
        msg = f"BACKUP FAILED at epoch {completed_epoch}: {result.stderr.strip()[:300]}"
        print(f"\n{'='*70}\n{msg}\n{'='*70}\n")
        _backup_failures.append((completed_epoch, result.stderr.strip()))


def relay_callback(trainer):
    completed = trainer.epoch + 1
    if _session_start_epoch["value"] is None:
        _session_start_epoch["value"] = completed - 1
    epochs_this_session = completed - _session_start_epoch["value"]
    is_budget_stop = epochs_this_session >= SESSION_EPOCH_BUDGET
    is_periodic_backup = completed % BACKUP_EVERY_N_EPOCHS == 0

    if is_budget_stop or is_periodic_backup:
        _push_backup(trainer, completed)
    if is_budget_stop:
        print(f"\nSession budget reached (total completed: {completed}/{trainer.epochs}). Stopping gracefully.")
        trainer.stop = True


t0 = time.time()
if resume_from is not None:
    model = YOLO(str(resume_from))
    model.add_callback("on_train_epoch_end", relay_callback)
    results = model.train(resume=True)
else:
    model = YOLO("yolo11s-seg.pt")
    model.add_callback("on_train_epoch_end", relay_callback)
    results = model.train(name=RUN_NAME, **TRAIN_ARGS)
wall = time.time() - t0
print(f"\nThis session: {wall/60:.1f} min")

if _backup_failures:
    print(f"\nWARNING: {len(_backup_failures)} backup attempt(s) failed. "
          f"Failed epochs: {[e for e, _ in _backup_failures]}")
else:
    print("\nAll backup attempts this session succeeded.")

Ultralytics 8.4.110 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.3, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/damage-seg.yaml, degrees=0.0, deterministic=True, device=, dfl=1.7, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo11s_seg_finetune, nb

## 7. Evaluate (same per-class format as the other runs, for direct comparison)

In [ ]:
m = model.val(data=SEG_DATA_YAML, split="test", imgsz=TRAIN_ARGS["imgsz"])
print("Box    mAP50:", float(m.box.map50), " mAP50-95:", float(m.box.map))
print("Mask   mAP50:", float(m.seg.map50), " mAP50-95:", float(m.seg.map))

import pandas as pd
rows = []
for idx, ci in enumerate(m.box.ap_class_index):
    rows.append({
        "class": CLASS_NAMES[int(ci)],
        "box_mAP50": float(m.box.ap50[idx]),
        "mask_mAP50": float(m.seg.ap50[idx]),
    })
pd.DataFrame(rows).sort_values("mask_mAP50", ascending=False)